# CRISP reproduction on Google Colab

The same pipeline `scripts/reproduce.sh` runs on a laptop — fetch datasets → evaluate the
original model → train CRISP → train RMU → train ELM → write the comparison table — but on
a CUDA GPU instead of Apple silicon.

**Before you start:** `Runtime → Change runtime type → GPU`. Which GPU you are assigned
decides what is actually runnable:

| Runtime | VRAM | What fits |
| --- | --- | --- |
| T4 (free tier) | ~15 GB | `configs/smoke.yaml`; `gemma2-2b` eval, and training only tightly |
| L4 (Pro) | ~22 GB | full `gemma2-2b_bio` / `gemma2-2b_cyber` in bf16 — **the recommended target** |
| A100 40 GB (Pro+) | 40 GB | as above comfortably; `llama31-8b` is possible but tight |

T4 is Turing and has no native bfloat16. Cell 6 picks float32 there: this project trains
without a gradient scaler, so float16 would risk silent NaNs rather than a clean OOM.

**Colab disconnects.** Cell 4 is optional but strongly recommended — it puts the Hugging
Face cache, `data/` and `artifacts/` on your Drive, so a dropped session resumes instead of
re-downloading several GB of weights and redoing finished stages.

## 0. What to run

Everything below keys off these two variables.

`gemma2-2b_cyber` is the easier first run: its corpora are fully public, whereas the bio
forget corpus is gated behind a manual approval that can take days. Both use the same
public Gemma Scope SAEs.

In [ ]:
# configs/gemma2-2b_bio.yaml     headline reproduction; gated bio forget corpus
# configs/gemma2-2b_cyber.yaml   fully public corpora; runs on a fresh account today
# configs/llama31-8b_bio.yaml    needs an A100 40GB, and is tight even there
CONFIG = "configs/gemma2-2b_cyber.yaml"

# Subset of original,crisp,rmu,elm. "original,crisp" is the headline comparison
# and roughly half the wall clock of the full table.
STAGES = "original,crisp"

## 1. Check the GPU you were assigned

In [ ]:
import subprocess, sys

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or
      "no GPU visible -- Runtime > Change runtime type > GPU")
print("python", sys.version)

## 2. Get the code

If your fork is private, clone with `https://<user>:<token>@github.com/...` using a PAT
stored in Colab secrets instead.

In [ ]:
import os, pathlib, subprocess

REPO_URL = "https://github.com/sagnikc395/crisp-reproducibility.git"
ROOT = pathlib.Path("/content/crisp-reproducibility")

if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=False)

os.chdir(ROOT)
# The package declares requires-python >= 3.13, which Colab may not satisfy, so
# crisp is imported from src/ rather than pip-installed (see cell 5).
os.environ["PYTHONPATH"] = str(ROOT / "src")
print("cwd:", os.getcwd())

## 3. Credentials

Add this in the Colab sidebar (🔑 **Secrets**), with *Notebook access* enabled:

| Secret | Needed for |
| --- | --- |
| `HF_TOKEN` | the gated `google/gemma-2-2b` weights, and the gated bio forget corpus |

The Fluency / Concept judge needs no key — it is a local `Qwen/Qwen3-4B-Thinking-2507`
downloaded on first use (~8 GB), unloaded again once scoring finishes. Pass `--no-judge`
to skip it.

Accept the Gemma licence on its model page (and, for the bio config, request access to
`cais/wmdp-bio-forget-corpus`) with the *same* account the token belongs to. This cell
writes the `.env` at the repo root that `crisp.utils.load_dotenv` reads.

In [ ]:
from google.colab import userdata

lines = []
for key in ("HF_TOKEN",):
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if value:
        lines.append(f"{key}={value}")
        os.environ[key] = value
        print(f"{key}: set")
    else:
        print(f"{key}: MISSING")

(ROOT / ".env").write_text("\n".join(lines) + ("\n" if lines else ""))
if "HF_TOKEN" not in os.environ:
    print("\nwithout HF_TOKEN only configs/smoke.yaml will run")

## 4. Persist across disconnects (optional, recommended)

Redirects the HF cache, `data/` and `artifacts/` onto Drive. Because `reproduce.sh` skips
any stage whose `artifacts/results/<run>__<split>.json` already exists, a session that dies
mid-run picks up where it stopped when you re-run the notebook.

Skip this cell to keep everything in ephemeral local storage — faster I/O, but you lose it
all on disconnect.

In [ ]:
USE_DRIVE = True  # set False to keep everything local/ephemeral

if USE_DRIVE:
    import shutil
    from google.colab import drive

    drive.mount("/content/drive")
    STORE = pathlib.Path("/content/drive/MyDrive/crisp")

    # Weights and SAEs are the expensive download; keep them out of the repo tree.
    os.environ["HF_HOME"] = str(STORE / "hf_cache")
    (STORE / "hf_cache").mkdir(parents=True, exist_ok=True)

    def link_to_drive(name: str) -> None:
        """Replace ROOT/<name> with a symlink into Drive, preserving what is there."""
        local, remote = ROOT / name, STORE / name
        if local.is_symlink():
            return
        remote.mkdir(parents=True, exist_ok=True)
        if local.exists():
            for item in local.iterdir():
                target = remote / item.name
                if not target.exists():
                    shutil.move(str(item), str(target))
            shutil.rmtree(local)
        local.symlink_to(remote, target_is_directory=True)

    for name in ("data", "artifacts"):
        link_to_drive(name)
    print("persisting to", STORE)
else:
    print("ephemeral storage: everything is lost when this runtime ends")

## 5. Install dependencies

Deliberately **not** `pip install -e .`. The project requires Python ≥ 3.13, which Colab
may not provide, and resolving `torch>=2.13` would replace Colab's CUDA-matched torch build
with a multi-GB download. So the runtime dependencies are installed around the preinstalled
torch, and `crisp` is imported from `src/` via the `PYTHONPATH` set in cell 2.

Colab may ask you to restart the session afterwards. If it does: restart, re-run cells 0–4
(they are idempotent), then continue from the import check below.

In [ ]:
%pip install -q -U \
  "transformers>=5.14.1" "datasets>=5.0.0" "huggingface-hub>=0.30.0" \
  "accelerate>=1.0.0" "peft>=0.19.1" "safetensors>=0.4.5" \
  "tokenizers>=0.22.2" "pyyaml>=6.0" "tqdm>=4.66.0" "numpy>=2.0.0"

In [ ]:
import torch

sys.path.insert(0, str(ROOT / "src"))
import crisp  # noqa: F401

print("crisp imports ok | torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 6. Pick a preset for this GPU

The Colab counterpart of `reproduce.sh --local`. It sets:

- **dtype** — bf16 on Ampere and newer (compute capability ≥ 8.0), float32 on T4.
- **`eval.mmlu_max_per_subject`** — the general-MMLU utility column is ~14k questions and
  otherwise dominates every stage. Subsampling to 2 per subject keeps all 57 subjects
  represented at ~1% of the cost. WMDP and the in-domain MMLU columns — the ones the
  paper's claims rest on — always stay at full size. Set `FULL_MMLU = True` for the
  untruncated column and expect several extra hours.

In [ ]:
FULL_MMLU = False

OVERRIDES = []
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    major, _ = torch.cuda.get_device_capability(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    # Turing (T4) has no native bf16, and training here has no gradient scaler,
    # so float16 would risk NaNs. float32 fails loudly with OOM instead.
    dtype = "bfloat16" if major >= 8 else "float32"
    OVERRIDES += ["-o", "model.device=cuda", "-o", f"model.dtype={dtype}"]
    print(f"{gpu} | {vram:.0f} GB | sm_{major}x -> dtype={dtype}")
    if major < 8:
        print("note: gemma-2-2b in float32 is ~10.4 GB of weights plus ~1.8 GB of SAEs.\n"
              "      If this OOMs: -o model.sae_layers=[4,8,12], -o train.batch_size=1,\n"
              "      or run --stages original only. An L4 runtime avoids the problem.")
else:
    print("no CUDA -- only configs/smoke.yaml is realistic on CPU")

if not FULL_MMLU:
    OVERRIDES += ["-o", "eval.mmlu_max_per_subject=2"]

print("overrides:", " ".join(OVERRIDES))

## 7. Smoke test (~1 minute)

Tiny random model, no gated downloads. Run this before committing GPU-hours — it exercises
fetch → select → train → eval → report end to end.

In [ ]:
!bash scripts/reproduce.sh configs/smoke.yaml

## 8. Fetch the real datasets

Materialises `data/wmdp/*.jsonl`, `data/mcq/*.jsonl` and `data/MANIFEST.json`, so the
reproduction is pinned to files on disk rather than to whatever the Hub serves that day.
The domain is read out of `CONFIG`. `reproduce.sh` calls this itself; running it separately
just surfaces any gated-access failure before the long job starts.

In [ ]:
import yaml

cfg = yaml.safe_load((ROOT / CONFIG).read_text()) or {}
DOMAIN = (cfg.get("data") or {}).get("domain", "bio")
print(f"{CONFIG} -> domain={DOMAIN}, model={(cfg.get('model') or {}).get('name')}")

subprocess.run(["python", "-m", "crisp", "fetch", "--domain", DOMAIN],
               env=os.environ, check=False)

## 9. The full run

Rough wall-clock on an L4 with the subsampled MMLU column: **3–5 hours** for all four
stages, of which `original,crisp` is about half. That is past the point where free-tier
Colab tends to disconnect, so either

- run it in slices by narrowing `STAGES` in cell 0 — each stage checkpoints its result to
  `artifacts/results/`, and a re-run skips whatever is already there, or
- keep cell 4's Drive persistence on and simply re-run this cell after a disconnect.

Add `--fresh` only if you want to redo stages that already have results.

In [ ]:
cmd = ["bash", "scripts/reproduce.sh", CONFIG, "--stages", STAGES, *OVERRIDES]
print(" ".join(cmd), "\n")

# Streamed rather than !-escaped so the long log appears live, and so OVERRIDES
# reach argv without shell quoting games.
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1, env=os.environ) as proc:
    for line in proc.stdout:
        print(line, end="")
print("\nexit code:", proc.returncode)

## 10. The results table

In [ ]:
from IPython.display import Markdown, display

subprocess.run(["python", "-m", "crisp", "report"], env=os.environ, check=False)

table = ROOT / "artifacts" / "results" / "README.md"
display(Markdown(table.read_text() if table.exists() else "no results yet"))

In [ ]:
# Pull the results off the runtime before it disappears.
# Unnecessary if cell 4 already symlinked artifacts/ into Drive.
from google.colab import files

subprocess.run(["zip", "-r", "/content/crisp_results.zip", "artifacts/results"], check=False)
files.download("/content/crisp_results.zip")

## Troubleshooting

| Symptom | Cause / fix |
| --- | --- |
| `preflight: HF_TOKEN cannot access google/gemma-2-2b` | licence not accepted, or the token belongs to a different account than the one that accepted it |
| `fetch` fails on the bio forget corpus | `cais/wmdp-bio-forget-corpus` is gated behind manual approval. Switch `CONFIG` to `configs/gemma2-2b_cyber.yaml` meanwhile, or point at your own file with `-o data.target_corpus=...` |
| `CUDA out of memory` | a T4 in float32. Try `-o model.sae_layers=[4,8,12]`, `-o train.batch_size=1`, `-o selection.batch_size=2`, or switch to an L4 runtime |
| loss goes to `nan` | almost always float16 on a pre-Ampere GPU — this codebase trains without a gradient scaler. Use float32 |
| Fluency / Concept columns blank | run with `--no-judge`, or the rater was truncated — check the log for `ratings unparsed` and raise `-o eval.judge_max_new_tokens=4096` |
| `CUDA out of memory` during `judge` | the 4B rater loads next to the model under test. Lower `-o eval.judge_batch_size=1` |
| session died mid-run | re-run cells 0–6, then cell 9. Finished stages are skipped |
| `ModuleNotFoundError: crisp` | the runtime restarted after the pip install and lost `PYTHONPATH`. Re-run cell 2 |